# Preprocessing teks Indonesia · satu metode, satu contoh

**Setiap metode memakai satu contoh sintetis yang berdiri sendiri.** Hasil metode sebelumnya
tidak diteruskan ke contoh berikutnya. Dengan begitu, before/after menunjukkan perubahan
dari metode yang sedang dibahas saja. Semua output sudah dijalankan dan tersimpan.

Polanya selalu: **tujuan → fungsi pendek → satu input → satu output**. Tidak membaca CSV MBG
atau menampilkan sampel komentar nyata. Data aslinya tetap tersedia untuk pemakaian pipeline nanti.

| Bagian | Metode |
|---|---|
| A | Validasi teks, byte escape, encoding, Unicode, HTML/entity, invisible, whitespace |
| B | Obfuscation, URL, email, telepon, mention, hashtag, lowercase |
| C | Slang, elongation, tawa, reduplikasi, bahasa daerah, typo, repetisi punctuation |
| D | Emoji/emoticon, angka, punctuation, repetisi kata, stopword, stemming |
| E | Deskripsi emoji, resep IndoBERTweet, tokenisasi |
| F | Menggabungkan fungsi dan memakai ulang pada data baru |

Metode yang agresif tetap diperlihatkan agar efeknya jelas, bukan berarti semuanya wajib
diaktifkan. Negasi, angka, nama, emoji, dan bahasa daerah dapat membawa makna.
Versi sebelumnya dengan output 57.362 komentar nyata disimpan di
`legacy/Preprocessing_mbg_real.ipynb`.

## Persiapan

Jalankan dari atas ke bawah. Dependensi inti contoh: pandas, ftfy, emoji, beautifulsoup4,
Sastrawi. Instal sekali memakai `python -m pip install -r requirements.txt`, lalu dapat dipakai
offline. Tidak ada instalasi atau unduhan otomatis di notebook.

Sel berikut menyiapkan pola dan kamus kecil yang dipakai beberapa fungsi. Mapping demo
dibuat eksplisit supaya hasil mudah ditulis ulang; bukan kamus bahasa Indonesia lengkap.

In [1]:
"""Text-only preprocessing in small deterministic steps; also embedded in the notebook.

API: preprocess_texts(series, slang=None, options=None). Input order/index are preserved.
No downloads, no fitting, no record deletion. Each stage can be copied independently.
"""
import html
import re
import unicodedata
from functools import lru_cache

import emoji
import ftfy
import pandas as pd
from pathlib import Path
import json
from IPython.display import display

In [2]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
STOPWORDS = set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split()) - NEGATIONS - PROTECTED
REGIONAL = {'ora': 'tidak', 'teu': 'tidak', 'indak': 'tidak', 'apik': 'bagus'}
OBFUSCATION = {'k3r4cunan': 'keracunan', 'k3r4cun4n': 'keracunan',
               'ker@cunan': 'keracunan', 'koru*si': 'korupsi'}
# URL/email/account/hashtag are single tokens: normalization never enters them.
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)


def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

SLANG_USED = dict(SLANG)
TYPO_APPROVED = {'maknan': 'makanan', 'bergzi': 'bergizi'}  # mapping khusus contoh
HASHTAG_APPROVED = {}
example_log = []

def show_result(method, before, after):
    example_log.append({'method': method, 'before': repr(before), 'after': repr(after)})
    display(pd.DataFrame({'before': [repr(before)], 'after': [repr(after)]}))

## A · Cleaning teknis

### A1 · Validasi tipe teks

Null/angka/list menjadi string kosong pada teks kerja, bukan literal `nan` atau `123`. String asli `"nan"`/`"None"` tetap teks. Baris tidak dibuang; nilai sumber tetap ada di raw.

**Satu contoh independen:**

In [3]:
def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

before = None
after = valid_text(before)
show_result('A1 · Validasi tipe teks', before, after)

,before,after
0,None,''


### A2 · Decode byte escape

Kembalikan escape byte UTF-8 literal menjadi emoji. Byte invalid tetap utuh. Jangan melakukan decode unicode_escape pada seluruh teks.

**Satu contoh independen:**

In [4]:
def decode_byte_escapes(text):
    def decode(match):
        try:
            return bytes.fromhex(match[0].replace('\\x', '')).decode('utf-8')
        except UnicodeDecodeError:
            return match[0]  # invalid bytes remain visible for review
    return re.sub(r'(?:\\x[0-9a-fA-F]{2})+', decode, text)

before = 'emoji \\xf0\\x9f\\x98\\xa1'
after = decode_byte_escapes(before)
show_result('A2 · Decode byte escape', before, after)

,before,after
0,'emoji \\xf0\\x9f\\x98\\xa1','emoji 😡'


### A3 · Perbaiki encoding rusak

ftfy memperbaiki mojibake. Perubahan tipografi, lebar karakter, dan HTML di ftfy dimatikan agar setiap transformasi terlihat pada tahapnya sendiri.

**Satu contoh independen:**

In [5]:
def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

before = 'cafÃ©, nasi sehat'
after = repair_encoding(before)
show_result('A3 · Perbaiki encoding rusak', before, after)

,before,after
0,"'cafÃ©, nasi sehat'","'café, nasi sehat'"


### A4 · Normalisasi Unicode NFC

Samakan representasi kanonis. Tidak menghapus aksara daerah/emoji dan tidak memaksa ASCII/NFKC.

**Satu contoh independen:**

In [6]:
def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

before = 'Café'
after = normalize_unicode(before)
show_result('A4 · Normalisasi Unicode NFC', before, after)

,before,after
0,'Café','Café'


Pada contoh Unicode, `é` dapat tampak sama di layar. Input menggunakan `e + U+0301`; output NFC menggunakan satu karakter `U+00E9`.

### A5 · Buang markup HTML bila ada

Gunakan parser hanya bila input benar-benar markup HTML. Script/style dibuang. Plain text seperti `nilai < 5 > 1` jangan diparse sembarangan.

**Satu contoh independen:**

In [7]:
def remove_html(text):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    return soup.get_text(' ')

before = '<p>Menu <b>sehat</b></p><script>bad()</script>'
after = remove_html(before)
show_result('A5 · Buang markup HTML bila ada', before, after)

,before,after
0,'<p>Menu <b>sehat</b></p><script>bad()</script>','Menu sehat'


### A6 · Decode HTML entities

`&amp;`, `&quot;`, `&#...;` menjadi karakter aslinya. Satu lapis decoding; tidak berulang tanpa batas pada teks yang sengaja menuliskan entity.

**Satu contoh independen:**

In [8]:
def decode_html_entities(text):
    return html.unescape(text)

before = 'MBG &amp; gizi &quot;baik&quot;'
after = decode_html_entities(before)
show_result('A6 · Decode HTML entities', before, after)

,before,after
0,'MBG &amp; gizi &quot;baik&quot;',"'MBG & gizi ""baik""'"


### A7 · Bersihkan karakter kontrol

Hapus BOM, zero-width space, word joiner, null byte; kontrol lain menjadi spasi. ZWJ/ZWNJ dan variation selectors dipertahankan agar emoji gabungan tidak pecah.

**Satu contoh independen:**

In [9]:
def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    # ZWJ, ZWNJ and variation selectors remain: preserve emoji and valid writing.

before = 'teks\u200bbaru\x00 👩\u200d🍳'
after = remove_invisible(before)
show_result('A7 · Bersihkan karakter kontrol', before, after)

,before,after
0,'teks\u200bbaru\x00 👩\u200d🍳','teksbaru 👩\u200d🍳'


### A8 · Rapikan whitespace

Gabungkan tab/newline/spasi berulang dan strip tepi. Notebook ini untuk komentar satu baris; untuk dokumen berparagraf, pakai versi yang mempertahankan newline.

**Satu contoh independen:**

In [10]:
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

before = '  menu\tsehat\r\n  merata  '
after = normalize_whitespace(before)
show_result('A8 · Rapikan whitespace', before, after)

,before,after
0,' menu\tsehat\r\n merata ','menu sehat merata'


## B · Elemen media sosial

### B1 · Normalisasi teks tersamarkan

Hanya allowlist kata yang eksplisit: tidak mengubah semua 3→e/4→a karena nomor dan kode bisa rusak. Dilakukan sebelum mention supaya `ker@cunan` bisa dikenali sebagai kata.

**Satu contoh independen:**

In [11]:
def normalize_obfuscation(text, mapping=OBFUSCATION):
    return re.sub(r'(?<!\w)[a-zA-Z][a-zA-Z0-9@*$]{3,}(?!\w)',
                  lambda m: mapping.get(m[0].casefold(), m[0]), text)

before = 'k3r4cUn4n ker@cunan koru*si S2 Rp10.000'
after = normalize_obfuscation(before)
show_result('B1 · Normalisasi teks tersamarkan', before, after)

,before,after
0,'k3r4cUn4n ker@cunan koru*si S2 Rp10.000','keracunan keracunan korupsi S2 Rp10.000'


### B2 · Ganti URL

URL panjang menjadi placeholder. Tanda baca penutup dipertahankan dengan heuristik sederhana; raw tetap menyimpan tautan lengkap.

**Satu contoh independen:**

In [12]:
def replace_urls(text):
    # Preserve terminal sentence punctuation; a heuristic, not a complete URL parser.
    return URL.sub(lambda m: ' URL ' + m[0][len(m[0].rstrip('.,!?;)')):], text)

before = 'cek https://contoh.id/x?q=gk, sekarang!'
after = replace_urls(before)
show_result('B2 · Ganti URL', before, after)

,before,after
0,"'cek https://contoh.id/x?q=gk, sekarang!'","'cek URL , sekarang!'"


### B3 · Ganti alamat email

Email sebelum mention agar `@domain` tidak dianggap akun.

**Satu contoh independen:**

In [13]:
def replace_emails(text):
    return re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', ' EMAIL ', text)

before = 'kirim admin@contoh.id ya'
after = replace_emails(before)
show_result('B3 · Ganti alamat email', before, after)

,before,after
0,'kirim admin@contoh.id ya','kirim EMAIL ya'


### B4 · Ganti nomor telepon

Pola konservatif nomor seluler Indonesia. Nominal/tanggal tidak diganti. Format telepon di luar pola perlu aturan tambahan.

**Satu contoh independen:**

In [14]:
def replace_phones(text):
    return re.sub(r'(?<!\w)(?:\+62|62|0)[ -]?8(?:[ -]?\d){8,11}(?!\w)', ' PHONE ', text)

before = 'WA +62 812-3456-7890, harga Rp10.000'
after = replace_phones(before)
show_result('B4 · Ganti nomor telepon', before, after)

,before,after
0,"'WA +62 812-3456-7890, harga Rp10.000'","'WA PHONE , harga Rp10.000'"


### B5 · Ganti mention

Mention menjadi placeholder. Nama orang di kalimat biasa tetap ada.

**Satu contoh independen:**

In [15]:
def replace_mentions(text):
    return re.sub(r'(?<![\w@])@[\w.]+', ' USER ', text)

before = '@budi dan @siti.id bahas MBG'
after = replace_mentions(before)
show_result('B5 · Ganti mention', before, after)

,before,after
0,'@budi dan @siti.id bahas MBG',' USER dan USER bahas MBG'


### B6 · Segmentasi hashtag

Pisahkan CamelCase/akronim, pertahankan isi kata. Hashtag lowercase gabungan hanya dipecah lewat mapping yang disetujui.

**Satu contoh independen:**

In [16]:
def segment_hashtags(text, mapping=None):
    mapping = {} if mapping is None else mapping
    def split(match):
        word = match[1]
        # #tolakmbg stays tolakmbg unless an explicit reviewed map is supplied.
        return ' ' + mapping.get(word.casefold(), re.sub(
            r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word)) + ' '
    return re.sub(r'(?<!\w)#(\w+)', split, text)

before = '#TolakMBG #DukungProgramMBG #tolakmbg'
after = (lambda s: segment_hashtags(s, HASHTAG_APPROVED))(before)
show_result('B6 · Segmentasi hashtag', before, after)

,before,after
0,'#TolakMBG #DukungProgramMBG #tolakmbg',' Tolak MBG Dukung Program MBG tolakmbg '


### B7 · Lowercase

Casefold untuk cabang normalisasi/clean. Kapitalisasi asli tetap tersedia di safe/raw untuk model yang membutuhkannya.

**Satu contoh independen:**

In [17]:
def lowercase(text):
    return text.casefold()

before = 'TIDAK Bagus! SPPG MBG'
after = lowercase(before)
show_result('B7 · Lowercase', before, after)

,before,after
0,'TIDAK Bagus! SPPG MBG','tidak bagus! sppg mbg'


## C · Normalisasi kata dan ekspresi

### C1 · Normalisasi slang dan singkatan informal

Exact-token mapping, bukan replace substring. `gk,` tetap dikenali, isi URL/email/hashtag tidak diubah. `sm/dr/bs` tidak ditebak. Akronim MBG/SPPG/BGN tidak diekspansi otomatis.

**Satu contoh independen:**

In [18]:
def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

before = 'mbg bgs bgt, tp gk merata sm dr SPPG'
after = (lambda s: normalize_slang(s, SLANG_USED))(before)
show_result('C1 · Normalisasi slang dan singkatan informal', before, after)

,before,after
0,"'mbg bgs bgt, tp gk merata sm dr SPPG'","'mbg bagus banget, tapi tidak merata sm dr SPPG'"


### C2 · Normalisasi huruf memanjang

Kurangi pengulangan 3+ hanya jika hasil ada di VOCAB. Double letter sah seperti massa/Allah tetap utuh; kata yang belum dikenal tidak dipaksa. Perluas VOCAB dari sumber yang dikurasi.

**Satu contoh independen:**

In [19]:
def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

before = 'baguuuuus bangeeet Allah massa xyztttt'
after = normalize_elongation(before)
show_result('C2 · Normalisasi huruf memanjang', before, after)

,before,after
0,'baguuuuus bangeeet Allah massa xyztttt','bagus banget Allah massa xyztttt'


### C3 · Normalisasi tawa

Kelompokkan variasi pola wkwk/haha yang eksplisit. Tawa tetap token ekspresi, tidak dibuang.

**Satu contoh independen:**

In [20]:
def normalize_laughter(text):
    def replace(word):
        return 'wkwk' if re.fullmatch(r'(?:wk){2,}|w+k+w+k+[wk]*', word, re.I) else (
            'haha' if re.fullmatch(r'(?:ha){2,}h?', word, re.I) else word)
    return map_words(text, replace)

before = 'wkwkwkwkwk hahahaha bagus'
after = normalize_laughter(before)
show_result('C3 · Normalisasi tawa', before, after)

,before,after
0,'wkwkwkwkwk hahahaha bagus','wkwk haha bagus'


### C4 · Normalisasi reduplikasi angka 2

Whitelist `anak2/orang2/teman2`, bukan semua token berangka. S2/GPT2/angka tanggal tetap.

**Satu contoh independen:**

In [21]:
def normalize_reduplication(text, mapping=None):
    # A whitelist avoids turning grade S2, model GPT2, dates or quantities into words.
    mapping = {'anak2': 'anak-anak', 'orang2': 'orang-orang', 'teman2': 'teman-teman'} if mapping is None else mapping
    return re.sub(r'\b\w+2\b', lambda m: mapping.get(m[0].casefold(), m[0]), text)

before = 'anak2 orang2 teman2 S2 GPT2'
after = normalize_reduplication(before)
show_result('C4 · Normalisasi reduplikasi angka 2', before, after)

,before,after
0,'anak2 orang2 teman2 S2 GPT2','anak-anak orang-orang teman-teman S2 GPT2'


### C5 · Normalisasi bahasa daerah opsional

Contoh mapping daerah yang eksplisit, bukan penerjemah otomatis. Pada data baru, aktifkan hanya setelah memeriksa konteks.

**Satu contoh independen:**

In [22]:
def normalize_regional(text, mapping=REGIONAL):
    return map_words(text, lambda w: mapping.get(w.casefold(), w))

before = 'ora apik dan teu enak'
after = normalize_regional(before)
show_result('C5 · Normalisasi bahasa daerah opsional', before, after)

,before,after
0,'ora apik dan teu enak','tidak bagus dan tidak enak'


### C6 · Terapkan typo yang sudah direview

Terapkan hanya pasangan typo yang sudah diperiksa. Contoh ini memakai mapping `maknan→makanan` dan `bergzi→bergizi`; bukan fuzzy correction semua kata tak dikenal.

**Satu contoh independen:**

In [23]:
def apply_reviewed_typos(text, mapping=None):
    mapping = {} if mapping is None else mapping
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

before = 'maknan bergzi SPPG'
after = (lambda s: apply_reviewed_typos(s, TYPO_APPROVED))(before)
show_result('C6 · Terapkan typo yang sudah direview', before, after)

,before,after
0,'maknan bergzi SPPG','makanan bergizi SPPG'


### C7 · Ringkas tanda baca berulang

Pengulangan ekstrem jadi maksimal dua: `!!!!!→!!`. Jenis tanda baca tidak disamakan. Cabang clean di bawah akan membuang tanda bacanya sepenuhnya.

**Satu contoh independen:**

In [24]:
def normalize_punctuation_repeat(text):
    return re.sub(r'([!?.,])\1{2,}', r'\1\1', text)

before = 'serius????!!!! bagus.....'
after = normalize_punctuation_repeat(before)
show_result('C7 · Ringkas tanda baca berulang', before, after)

,before,after
0,'serius????!!!! bagus.....','serius??!! bagus..'


## D · Cleaning agresif

### D1 · Hapus emoji

Hapus satu sequence emoji secara utuh, termasuk emoji keluarga/skin tone/flag. Alternatif deskripsi emoji tersedia terpisah di bagian E.

**Satu contoh independen:**

In [25]:
def remove_emoji(text):
    return emoji.replace_emoji(text, replace=' ')

before = 'bagus 😍 🙏🏽 👨\u200d👩\u200d👧\u200d👦 🇮🇩'
after = remove_emoji(before)
show_result('D1 · Hapus emoji', before, after)

,before,after
0,'bagus 😍 🙏🏽 👨\u200d👩\u200d👧\u200d👦 🇮🇩','bagus '


### D2 · Hapus emoticon ASCII

Aturan umum :) :( :)) <3, bukan semua emoticon yang pernah ada. Tanda baca sisanya dibersihkan di D4.

**Satu contoh independen:**

In [26]:
def remove_emoticons(text):
    return re.sub(r"(?<!\w)(?:[:;=8][\-^']?[)(/DPp]+|<3)(?!\w)", ' ', text)

before = 'bagus :)) sedih :( cinta <3'
after = remove_emoticons(before)
show_result('D2 · Hapus emoticon ASCII', before, after)

,before,after
0,'bagus :)) sedih :( cinta <3','bagus sedih cinta '


### D3 · Hapus angka opsional

Contoh efek penghapusan angka. Nominal/tanggal akan kehilangan informasi; default API tidak mengaktifkannya. Angka di kode alfanumerik tetap.

**Satu contoh independen:**

In [27]:
def remove_numbers(text):
    # Opt-in destructive view: amounts/dates become incomplete, inspect before using.
    return re.sub(r'(?<!\w)\d+(?:[.,:/-]\d+)*(?!\w)', ' ', text)

before = '3 siswa Rp10.000 pada 10/11/2026 S2'
after = remove_numbers(before)
show_result('D3 · Hapus angka opsional', before, after)

,before,after
0,'3 siswa Rp10.000 pada 10/11/2026 S2',' siswa Rp10. pada S2'


### D4 · Hapus tanda baca dan simbol

Ubah ke spasi agar kata tidak menempel. Huruf Unicode/angka tetap. Nama dengan tanda hubung berubah menjadi kata terpisah; periksa implikasinya untuk entitas.

**Satu contoh independen:**

In [28]:
def remove_punctuation(text):
    # Keep all scripts/letters/digits; punctuation becomes a SPACE, never concatenation.
    return re.sub(r'[^\w\s]|_', ' ', text)

before = 'anak-anak, bagus!!! tidak/buruk; 10.000'
after = remove_punctuation(before)
show_result('D4 · Hapus tanda baca dan simbol', before, after)

,before,after
0,"'anak-anak, bagus!!! tidak/buruk; 10.000'",'anak anak bagus tidak buruk 10 000'


### D5 · Ringkas repetisi kata berturut-turut

Hanya kata identik berurutan. Ekspresi `bagus bagus` akan jadi `bagus`; normalized masih menyimpan ekspresinya.

**Satu contoh independen:**

In [29]:
def reduce_repeated_words(text):
    # Aggressive only: even "bagus bagus" can be expressive, so retain another view.
    return re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text, flags=re.I)

before = 'bagus bagus bagus tapi tidak tidak'
after = reduce_repeated_words(before)
show_result('D5 · Ringkas repetisi kata berturut-turut', before, after)

,before,after
0,'bagus bagus bagus tapi tidak tidak','bagus tapi tidak'


### D6 · Hapus stopword

Daftar kecil yang transparan dan bisa diganti. Negasi dan akronim terlindungi tidak dibuang walaupun masuk daftar tambahan.

**Satu contoh independen:**

In [30]:
def remove_stopwords(text, words=STOPWORDS):
    blocked = set(words) - NEGATIONS - PROTECTED
    return ' '.join(w for w in text.split() if w.casefold() not in blocked)

before = 'makanan yang bagus untuk anak dan tidak merata'
after = remove_stopwords(before)
show_result('D6 · Hapus stopword', before, after)

,before,after
0,'makanan yang bagus untuk anak dan tidak merata','makanan bagus anak tidak merata'


### D7 · Stemming Sastrawi

Bentuk dasar per kata unik dengan cache terbatas. Negasi dan akronim terlindungi. Stemming bukan spellchecker dan dapat mengubah nama/istilah: safe/normalized tetap ada.

**Satu contoh independen:**

In [31]:
@lru_cache(maxsize=1)
def get_stemmer():
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    return StemmerFactory().create_stemmer()


@lru_cache(maxsize=50000)
def stem_word(word):
    return get_stemmer().stem(word)


def stem_text(text):
    return map_words(text, lambda w: w if w.casefold() in PROTECTED | NEGATIONS else stem_word(w))

before = 'makanan membantu pemerataan dan tidak bergizi SPPG'
after = stem_text(before)
show_result('D7 · Stemming Sastrawi', before, after)

,before,after
0,'makanan membantu pemerataan dan tidak bergizi...,'makan bantu perata dan tidak gizi SPPG'


## E · Representasi alternatif dan tokenisasi

### E1 · Emoji menjadi deskripsi

Shortcode Inggris dari library emoji, bukan hasil klasifikasi sentimen dan bukan terjemahan Indonesia.

**Satu contoh independen:**

In [32]:
def describe_emoji(text):
    return emoji.demojize(text, delimiters=(' ', ' '))

before = 'Menu 😍 👩\u200d🍳'
after = describe_emoji(before)
show_result('E1 · Emoji menjadi deskripsi', before, after)

,before,after
0,'Menu 😍 👩\u200d🍳','Menu smiling_face_with_heart-eyes woman_co...


### E2 · Resep khusus IndoBERTweet

Lowercase, mention→@USER, URL→HTTPURL, emoji shortcode mengikuti resep publik model. Jangan menambahkan stopword/stemming ke view ini tanpa eksperimen.

**Satu contoh independen:**

In [33]:
def indobertweet_view(text):
    text = URL.sub('HTTPURL', text.lower())
    text = re.sub(r'(?<![\w@])@[\w.]+', '@USER', text)
    return normalize_whitespace(emoji.demojize(text))

before = '@Budi MBG BAGUS 😍 https://contoh.id'
after = indobertweet_view(before)
show_result('E2 · Resep khusus IndoBERTweet', before, after)

,before,after
0,'@Budi MBG BAGUS 😍 https://contoh.id','@USER mbg bagus :smiling_face_with_heart-eyes...


### E3 · Tokenisasi sederhana

Kalimat menjadi list kata menggunakan pemisah spasi. Untuk model pretrained, gunakan tokenizer
model tersebut; ini hanya contoh tokenisasi transparan pada teks clean.

In [34]:
def tokenize(text):
    # Transparent baseline tokenization of the clean view; not a model tokenizer.
    return text.split()

before = 'menu sehat tidak merata'
after = tokenize(before)
show_result('E3 · Tokenisasi sederhana', before, after)

,before,after
0,'menu sehat tidak merata',"['menu', 'sehat', 'tidak', 'merata']"


## F · Gabungkan dan pakai ulang

Bagian A–E memperlihatkan metode secara terpisah. API berikut menyusun **fungsi yang sama**
menjadi cabang safe → normalized → clean. Tidak ada aturan preprocessing kedua yang ditulis ulang.
Urutan langkahnya dapat dilihat pada `build_steps`; opsi agresif diatur melalui `options`.

Contoh penggunaan di bawah memakai **satu komentar sintetis gabungan** untuk memperlihatkan
cara memanggil API. Ini bagian penggunaan pipeline, bukan contoh metode tambahan.

In [35]:
DEFAULT_OPTIONS = dict(html_input=False, escaped_bytes=False, regional=False,
                       remove_numbers=False, stem=True, stopwords=True,
                       slang=None, typos=None, hashtags=None)


def build_steps(options=None):
    unknown = set(options or {}) - set(DEFAULT_OPTIONS)
    if unknown:
        raise ValueError(f'Opsi tidak dikenal: {sorted(unknown)}')
    options = {**DEFAULT_OPTIONS, **(options or {})}
    safe = [('valid', valid_text)]
    if options['escaped_bytes']:
        safe.append(('escaped_bytes', decode_byte_escapes))
    safe += [('encoding', repair_encoding), ('unicode', normalize_unicode)]
    if options['html_input']:
        safe.append(('html', remove_html))
    safe += [('entities', decode_html_entities), ('invisible', remove_invisible), ('space', normalize_whitespace)]
    normalized = [('obfuscation', normalize_obfuscation), ('urls', replace_urls),
                  ('emails', replace_emails), ('phones', replace_phones), ('mentions', replace_mentions),
                  ('hashtags', lambda s: segment_hashtags(s, options['hashtags'])), ('lowercase', lowercase),
                  ('slang', lambda s: normalize_slang(s, SLANG if options['slang'] is None else options['slang'])),
                  ('elongation', normalize_elongation), ('laughter', normalize_laughter),
                  ('reduplication', normalize_reduplication)]
    if options['regional']:
        normalized.append(('regional', normalize_regional))
    normalized += [('typo', lambda s: apply_reviewed_typos(s, options['typos'])),
                   ('punctuation_repeat', normalize_punctuation_repeat), ('space', normalize_whitespace)]
    clean = [('emoji', remove_emoji), ('emoticons', remove_emoticons)]
    if options['remove_numbers']:
        clean.append(('numbers', remove_numbers))
    clean += [('punctuation', remove_punctuation), ('repeated_words', reduce_repeated_words)]
    if options['stopwords']:
        clean.append(('stopwords', remove_stopwords))
    if options['stem']:
        clean.append(('stemming', stem_text))
    clean.append(('space', normalize_whitespace))
    return dict(safe=safe, normalized=normalized, clean=clean)


def transform_unique(texts, function):
    # Invalid values handled separately, so unhashable input never becomes a cache key.
    texts = texts.map(valid_text) if function is valid_text else texts
    cache = {text: function(text) for text in pd.unique(texts)}
    return texts.map(cache)


def preprocess_texts(texts, options=None):
    """Same functions and order as notebook. No data-dependent fitting or randomness."""
    texts = texts.copy() if isinstance(texts, pd.Series) else pd.Series(texts)
    output = pd.DataFrame({'text_raw': texts})
    current = texts
    for branch, steps in build_steps(options).items():
        for _, function in steps:
            current = transform_unique(current, function)
        output['text_' + branch] = current
    output['text_emoji_described'] = output.text_normalized.map(describe_emoji)
    output['tokens'] = output.text_clean.map(tokenize)
    output['text_indobertweet'] = output.text_safe.map(indobertweet_view)
    return output

In [36]:
before = '  @budi MBG bgs bgt, tp gk merata!!! 😡 #TolakMBG  '
options = {'slang': SLANG_USED, 'stem': True, 'stopwords': True,
           'regional': False, 'remove_numbers': False, 'typos': {}}
pipeline_result = preprocess_texts(pd.Series([before]), options=options)
display(pipeline_result[['text_raw', 'text_safe', 'text_normalized', 'text_clean']])

# Verifikasi urutan sel manual dan API identik, pada satu input yang sama.
manual = pd.Series([before])
for branch, steps in build_steps(options).items():
    for _, function in steps:
        manual = transform_unique(manual, function)
    pd.testing.assert_series_equal(pipeline_result['text_' + branch], manual, check_names=False)
assert len(example_log) == 32
assert all(set(row) == {'method', 'before', 'after'} for row in example_log)
print('Lulus: 32 metode, masing-masing satu contoh; hasil pipeline manual = API.')

,text_raw,text_safe,text_normalized,text_clean
0,"@budi MBG bgs bgt, tp gk merata!!! 😡 #TolakM...","@budi MBG bgs bgt, tp gk merata!!! 😡 #TolakMBG","user mbg bagus banget, tapi tidak merata!! 😡 t...",user mbg bagus banget tapi tidak rata tolak mbg


Lulus: 32 metode, masing-masing satu contoh; hasil pipeline manual = API.


### Pakai pada data baru nanti

Kode ini sengaja tidak dijalankan oleh Run All. Tidak ada fallback atau pembacaan data nyata
tersembunyi di contoh metode. Salin fungsi yang diperlukan, atau gunakan API `stepwise.py`.

```python
from stepwise import preprocess_texts

df = pd.read_csv('data/MBG_PROCESSED.csv', dtype='string')
result = preprocess_texts(df['comment'], options={'stem': False})
```

Kamus luar dapat disiapkan lewat `lexicons.external_lexicons(...)`; sumber asli lokal berada di
`resources/downloaded/`. Mapping yang belum disetujui tidak otomatis diterapkan. Untuk memakai
kamus tinjauan: `options={'slang': kamus_yang_disetujui}`. Opsi `typos` pada data baru harus
berasal dari review, bukan menyalin mapping demo tanpa pemeriksaan.

Notebook contoh ini tidak menimpa ekspor data nyata pada `hasil_stepwise/`. Pengujian fungsi
tersedia di `tests/test_stepwise.py`, bukti eksekusi contoh di `reports/verification_examples.json`.

## Referensi dan sumber tambahan

Ditinjau 8 November 2026. Blueprint pengguna menjadi referensi kebutuhan; aturan notebook
membedakan cleaning aman, normalisasi, dan penghapusan agresif agar perubahan dapat diperiksa.

| Sumber primer | Pemakaian |
|---|---|
| [ftfy](https://ftfy.readthedocs.io/en/latest/config.html) | Repair encoding; transformasi lain dikendalikan eksplisit |
| [emoji](https://emoji-python.readthedocs.io/en/latest/api.html) | Hapus sequence utuh atau deskripsi sebagai view alternatif |
| [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) | Parser markup opt-in, bukan regex HTML |
| [Sastrawi](https://github.com/har07/PySastrawi) | Stemming per kata dengan cache pada cabang clean |
| [IndoBERTweet](https://github.com/indolem/IndoBERTweet) | Resep model: lowercase, @USER, HTTPURL, emoji shortcode |
| [indoNLP](https://hyuto.github.io/indo-nlp/api/preprocessing/) | Referensi normalisasi slang/elongation; fungsi lokal dibuat transparan |
| [Colloquial Lexicon](https://github.com/nasalsabila/kamus-alay) | Adapter CSV slang/formal, mapping luar butuh review |
| [Kamus Alay](https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection) | Adapter CSV tanpa header; konflik tidak diterapkan otomatis |
| [Combined Slang](https://github.com/louisowen6/NLP_bahasa_resources) | Adapter JSON dan pencatatan sumber/hash |

Tambahan di luar blueprint: [IndoCollex](https://aclanthology.org/2021.findings-acl.280/)
sebagai benchmark normalisasi informal, dan [normalisasi code-mixed Indonesia–Inggris](https://aclanthology.org/D19-5554/)
untuk mengevaluasi campuran bahasa secara terpisah. Ini rujukan eksperimen, bukan jaminan skor MBG.

Sumber opsional untuk pengembangan: [SALTIK](https://github.com/ir-nlp-csui/saltik),
[regional lexicon](https://data.mendeley.com/datasets/mrfc9mb2yt/1),
[parallel normalization corpus](https://github.com/ir-nlp-csui/parallel-corpus-for-lexical-normalization),
[BASA](https://github.com/Muanai/basa), dan [SymSpell](https://symspellpy.readthedocs.io/en/latest/examples/lookup.html).
Tidak dianggap sebagai normalizer siap pakai tanpa evaluasi; kosakata regional dan nama sering ambigu.

Fungsi non-teks pada API lama di preprocessing.py memakai
[datasketch](https://ekzhu.com/datasketch/lsh.html),
[RapidFuzz](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html), dan
[fastText](https://github.com/facebookresearch/fastText/blob/main/docs/language-identification.md).
Ketiganya tidak dijalankan pada alur notebook teks yang baru.

Kamus asli tidak dibundel untuk redistribusi. URL, SHA-256, mapping aktif, config, dan versi
library dicatat lokal. Periksa ketentuan sumber sebelum mendistribusikan resource.
Ablation pada data berlabel dan validasi yang sama diperlukan sebelum menyatakan skor meningkat.